# MGMT298D: Science and Strategy of AI
## Week 6: Convolutional Neural Networks
### UCLA Anderson School of Management

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load CIFAR-10 dataset and normalize to [0,1]
(x_train_full, y_train_full), (x_test, y_test) = datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Select balanced subset of 5000 training images
np.random.seed(42)
indices = []
for class_idx in range(10):
    class_indices = np.where(y_train_full.flatten() == class_idx)[0]
    indices.extend(np.random.choice(class_indices, 500, replace=False))
indices = np.array(indices)
x_train = x_train_full[indices]
y_train = y_train_full[indices]

print(f"Training set shape: {x_train.shape}")
print(f"Test set shape: {x_test.shape}")
print(f"Number of classes: 10")

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# Display one sample image per class in a 2x5 grid
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for class_idx in range(10):
    class_mask = y_train.flatten() == class_idx
    sample_idx = np.where(class_mask)[0][0]
    axes[class_idx].imshow(x_train[sample_idx])
    axes[class_idx].set_title(CLASS_NAMES[class_idx])
    axes[class_idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Build simple CNN: 1 conv block + fully connected layers
simple_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

simple_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_simple = simple_cnn.fit(x_train, y_train, epochs=10, batch_size=64, 
                                validation_split=0.1, verbose=0)

test_acc_simple = simple_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f"Simple CNN test accuracy: {test_acc_simple:.4f}")

In [ ]:
# Plot training and validation curves for simple CNN
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_simple.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history_simple.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Simple CNN: Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_simple.history['loss'], label='Train', linewidth=2)
axes[1].plot(history_simple.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Simple CNN: Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Build deeper CNN with 3 conv blocks
deeper_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

deeper_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_deeper = deeper_cnn.fit(x_train, y_train, epochs=10, batch_size=64, 
                                validation_split=0.1, verbose=0)

test_acc_deeper = deeper_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f"Deeper CNN test accuracy: {test_acc_deeper:.4f}")

In [ ]:
# Build regularized CNN with batch norm and dropout
regularized_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

regularized_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_regularized = regularized_cnn.fit(x_train, y_train, epochs=15, batch_size=64, 
                                          validation_split=0.1, verbose=0)

test_acc_regularized = regularized_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f"Regularized CNN test accuracy: {test_acc_regularized:.4f}")

In [ ]:
# Plot training and validation curves for regularized CNN
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_regularized.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history_regularized.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Regularized CNN: Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_regularized.history['loss'], label='Train', linewidth=2)
axes[1].plot(history_regularized.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Regularized CNN: Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Compare test accuracy across all three models
models_names = ['Simple CNN', 'Deeper CNN', 'Regularized CNN']
accuracies = [test_acc_simple, test_acc_deeper, test_acc_regularized]

plt.figure(figsize=(8, 5))
bars = plt.bar(models_names, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
plt.ylabel('Test Accuracy')
plt.title('Model Comparison')
plt.ylim([0, 1])
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, 
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Compute confusion matrix for regularized CNN
y_pred = np.argmax(regularized_cnn.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test.flatten(), y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, 
            yticklabels=CLASS_NAMES, cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: Regularized CNN')
plt.tight_layout()
plt.show()

In [ ]:
# Display predictions on test samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

predictions = regularized_cnn.predict(x_test[:10], verbose=0)
pred_labels = np.argmax(predictions, axis=1)

for i in range(10):
    axes[i].imshow(x_test[i])
    true_label = CLASS_NAMES[y_test[i, 0]]
    pred_label = CLASS_NAMES[pred_labels[i]]
    color = 'green' if pred_labels[i] == y_test[i, 0] else 'red'
    axes[i].set_title(f'Pred: {pred_label}\nTrue: {true_label}', color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Test different starting filter counts for simple 1-block CNN
filter_counts = [8, 16, 32, 64]
filter_accuracies = []

for filters in filter_counts:
    model = models.Sequential([
        Conv2D(filters, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    test_acc = model.evaluate(x_test, y_test, verbose=0)[1]
    filter_accuracies.append(test_acc)
    print(f"Filters={filters}: test accuracy = {test_acc:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(filter_counts, filter_accuracies, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Filters')
plt.ylabel('Test Accuracy')
plt.title('Effect of Filter Count on Simple CNN')
plt.grid(True, alpha=0.3)
plt.xticks(filter_counts)
plt.tight_layout()
plt.show()